In [ ]:
# Paths are configured via environment variables, see src/common/utils/src/utils/paths.py
import os
from utils.paths import DATA_ROOT, pkg_path

# Some funcations

In [ ]:
# usr/bin/env python3
import numpy as np
import os
import open3d as o3d
from scipy.spatial.transform import Rotation as R
import torch
# add this path to the python path
import sys
sys.path.append(pkg_path('networks') + '/src/networks')
from chamfer import chamfer_distance

def parse_translation_rotation(filepath):
    data = {'translation': {}, 'rotation': {}, 'name': []}
    current_category = None  # This will track whether we're filling translation or rotation

    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
            if 'translation:' in line:
                current_category = 'translation'
            elif 'rotation:' in line:
                current_category = 'rotation'
            else:
                # Check if we are in a translation or rotation block
                if current_category:
                    key_value = line.split(':')
                    if len(key_value) == 2:
                        key, value = key_value
                        key = key.strip()
                        value = value.strip()
                        # Convert the value to float if possible
                        try:
                            value = float(value)
                        except ValueError:
                            pass
                        data[current_category][key] = value
                        data['name'] = [os.path.basename(filepath)]
    return data

def pose_to_transformation(pose):
    """Convert a pose [x, y, z, qx, qy, qz, qw] to a 4x4 transformation matrix."""
    translation = pose[:3]
    quaternion = pose[3:]
    rotation_matrix = R.from_quat(quaternion).as_matrix()
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = translation
    return transformation_matrix

def unorganized_to_organized(points_xyzrgb, camera_intrinsics, image_width=640, image_height=480):
    """
    Convert unorganized point cloud (nx6) in camera frame back to organized format (hxwx6)
    
    Args:
        points_xyzrgb: (n, 6) array with xyz (in camera frame) and rgb
        camera_intrinsics: dict with 'fx', 'fy', 'cx', 'cy' or 3x3 matrix
        image_width: width of the output organized point cloud
        image_height: height of the output organized point cloud
    
    Returns:
        organized_pc: (height, width, 6) organized point cloud with invalid points as NaN
    """
    # Initialize organized point cloud with NaN
    organized_pc = np.full((image_height, image_width, 6), np.nan, dtype=np.float32)
    
    # Extract intrinsics
    if isinstance(camera_intrinsics, dict):
        fx = camera_intrinsics['fx']
        fy = camera_intrinsics['fy']
        cx = camera_intrinsics['cx']
        cy = camera_intrinsics['cy']
    else:
        fx = camera_intrinsics[0, 0]
        fy = camera_intrinsics[1, 1]
        cx = camera_intrinsics[0, 2]
        cy = camera_intrinsics[1, 2]
    
    # Points are already in camera frame
    xyz = points_xyzrgb[:, :3]
    rgb = points_xyzrgb[:, 3:]
    
    # Project to image plane
    # u = fx * x / z + cx
    # v = fy * y / z + cy
    valid_depth = xyz[:, 2] > 0
    u = (fx * xyz[:, 0] / xyz[:, 2] + cx).astype(int)
    v = (fy * xyz[:, 1] / xyz[:, 2] + cy).astype(int)
    
    # Filter points within image bounds
    valid_u = (u >= 0) & (u < image_width)
    valid_v = (v >= 0) & (v < image_height)
    valid_mask = valid_depth & valid_u & valid_v
    
    # Assign points to organized structure
    u_valid = u[valid_mask]
    v_valid = v[valid_mask]
    organized_pc[v_valid, u_valid, :3] = xyz[valid_mask]
    organized_pc[v_valid, u_valid, 3:] = rgb[valid_mask]
    
    return organized_pc

def cal_coverage(partial_pc, complete_pc, thr):
    '''
        partial_pc: 6 channels-xyzrgb.
    '''
    if not partial_pc.is_cuda:
        partial_pc = partial_pc.to('cuda')
    _, dist2 = get_chamfer_dists(partial_pc[..., 0:3], complete_pc)
    covered_points = dist2 < thr
    cov_ratio = torch.sum(covered_points).item() / covered_points.shape[1]
    return '%s/%s' % (str(torch.sum(covered_points).item()),
                        str(covered_points.shape[1])), cov_ratio

def get_chamfer_dists(pcd_x, pcd_y):
    # We compare the two sets of pointclouds by computing the chamfer loss.
    _, _, dist_x, dist_y = chamfer_distance(pcd_x, pcd_y)
    dist_x = dist_x.cpu().detach()
    dist_y = dist_y.cpu().detach()

    return dist_x, dist_y

def vis_PC(center, complete_pcd=None, partial_pcd = None, viewpoints_pcd= None, bbox= None):
    vis = o3d.visualization.Visualizer()
    vis.create_window()
    if partial_pcd is not None:
        vis.add_geometry(partial_pcd)
    if complete_pcd is not None:
        vis.add_geometry(complete_pcd)
    if viewpoints_pcd is not None:
        vis.add_geometry(viewpoints_pcd)
    if bbox is not None:
        vis.add_geometry(bbox)

    coord_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5, origin=[0, 0, 0])
    vis.add_geometry(coord_frame)
    ctr = vis.get_view_control()
    ctr.set_lookat(center)
    ctr.set_up([0, 0, 1])
    ctr.set_front([0, 1, 0])
    vis.run()

def downsample_point_cloud(pcd, bbox, voxel_size, neighbors=20, std_ratio=0.2):
    if bbox is not None:   
        pcd = pcd.crop(bbox)
    pcd.remove_non_finite_points()
    pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
    pcd_down_sampled, _ = pcd.remove_statistical_outlier(nb_neighbors=neighbors, std_ratio=std_ratio)
    return pcd_down_sampled
    

# Some parameters

In [ ]:
center = np.array([0.4, -0.8, 0.9]) 
size = np.array([0.6, 0.6, 1])
bbox = o3d.geometry.AxisAlignedBoundingBox(center-(size/2), center+(size/2))
bbox.color = (1, 0, 0) 
voxel_size = 0.003
view_index = 3

# Preliminary test if the pcd match the gt

In [ ]:
import numpy as np
import os
import open3d as o3d
from scipy.spatial.transform import Rotation as R

plant = 33
nbv_path = DATA_ROOT + '/real_world_data_new/real_nbv_data/plant%s/xyr_0_0_0'%str(plant)
gt_pcd_path = DATA_ROOT + '/real_world_data_new/real_gt_pcd/plant%s/tomato%s.ply'%(str(plant), str(plant))
gt_pcd = o3d.io.read_point_cloud(gt_pcd_path)

gt_pcd = downsample_point_cloud(gt_pcd, bbox=bbox, voxel_size=voxel_size)
gt_pcd.paint_uniform_color([0.5, 0.5, 0.5])  # Set grey
bbox = gt_pcd.get_axis_aligned_bounding_box()

#read all the txt files in the dir_path
txt_files = os.listdir(nbv_path)
txt_files = [f for f in txt_files if f.endswith('.txt')]

#read all the point clouds in the dir_path
pcd_files = os.listdir(nbv_path)
pcd_files = [f for f in pcd_files if f.endswith('.pcd')]

# transform all the point clouds using the translation and rotation in the txt files
acc_pcd = []
for txt_file in txt_files:
    path = os.path.join(nbv_path, txt_file)
    translation_rotation_data = parse_translation_rotation(path)
    translation = translation_rotation_data['translation']
    rotation = translation_rotation_data['rotation']
    name = translation_rotation_data['name'][0]

    # find the corresponding point cloud file
    pcd_name = name.split('_')[0] + '_b.pcd'
    if pcd_name not in pcd_files:
        continue  # skip if the point cloud file does not exist

    pcd_path = os.path.join(nbv_path, pcd_name)
    pcd = o3d.io.read_point_cloud(pcd_path)

    # create transformation matrix
    pose = [translation['x'], translation['y'], translation['z'], rotation['x'], rotation['y'], rotation['z'], rotation['w']]
    transformation_matrix = pose_to_transformation(pose)
    pcd.transform(transformation_matrix)

    pcd = downsample_point_cloud(pcd, bbox=bbox, voxel_size=voxel_size)
    acc_pcd.append(np.asarray(pcd.points))

acc_pcd = np.concatenate(acc_pcd, axis=0)
acc_pcd_ = o3d.geometry.PointCloud()
acc_pcd_.points = o3d.utility.Vector3dVector(acc_pcd)
acc_pcd_ = downsample_point_cloud(acc_pcd_, bbox=bbox, voxel_size=voxel_size)
print('Accumulated point cloud shape:', np.asarray(acc_pcd_.points).shape)

acc_pcd_.paint_uniform_color([0, 1, 0])  # Set color to green

# visualize the ground truth point cloud and the accumulated point cloud
o3d.visualization.draw_geometries([acc_pcd_, gt_pcd], window_name='Ground Truth and Accumulated Point Cloud', width=800, height=600)

# 1. Process partial point cloud data

## 1.1. Get viewpoints from .txt pose file

In [ ]:
## create viewpoints box#############################################################
%matplotlib inline
plant = 40
plant_data_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/plant%s/xyr_0_0_0'%str(plant)
txt_files = os.listdir(plant_data_dir)
txt_files = [f for f in txt_files if f.endswith('.txt')]

x_y_z_x_y_z_w = []
names = []
for text_file in txt_files:
    path = os.path.join(plant_data_dir, text_file)
    translation_rotation_data = parse_translation_rotation(path)
    try:
        x_y_z_x_y_z_w.append([translation_rotation_data['translation']['x'], translation_rotation_data['translation']['y'], translation_rotation_data['translation']['z'], translation_rotation_data['rotation']['x'], translation_rotation_data['rotation']['y'], translation_rotation_data['rotation']['z'], translation_rotation_data['rotation']['w']])
        names.append(translation_rotation_data['name'][0])
    except:
        print(text_file)

viewpoints = np.array(x_y_z_x_y_z_w, dtype=float)  # Ensure the array is of type float
names = np.array(names)
print(len(names))

min_x = np.min(viewpoints[:, 0])
max_x = np.max(viewpoints[:, 0])

min_z = np.min(viewpoints[:, 2])
max_z = np.max(viewpoints[:, 2])

x_values = np.linspace(min_x, max_x, 10)
z_values = np.linspace(min_z, max_z, 15)

# create meshgrid using x_values and z_values, and length
x, z = np.meshgrid(x_values, z_values)
x, z = x[..., np.newaxis], z[..., np.newaxis]

y, raw, pitch, yaw, w = np.zeros((15, 10, 1)), np.zeros((15, 10, 1)), np.zeros((15, 10, 1)), np.zeros((15, 10, 1)), np.zeros((15, 10, 1))
name = np.empty((15, 10, 1), dtype=object)
box = np.concatenate((x, y, z, raw, pitch, yaw, w, name), axis=-1)

for idx, viewpoint in enumerate(viewpoints):
    x, y, z, roll, pitch, yaw, w = viewpoint
    smallest_distance = 1000
    smallest_h = None
    smallest_w = None
    for h_index in range(15):
        for w_index in range(10):
            distance = abs(x - box[h_index, w_index, 0]) **2+abs(z - box[h_index, w_index, 2])**2
            if distance < smallest_distance:
                smallest_distance = distance
                smallest_h = h_index
                smallest_w = w_index
    box[smallest_h, smallest_w,:] = [x, y, z, roll, pitch, yaw, w, names[idx]]

views = box[..., :7]
print(views.shape)
# visualize the viewpoints using matplotlib using 3D scatter plot
import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(views[..., 0], views[..., 1], views[..., 2])
ax.set_xlim(-1, 0.5)
ax.set_ylim(-1, 0.5)
ax.set_zlim(1, 2.5)   

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

np.save(DATA_ROOT + '/real_world_data_new/real_nbv_data/viewpoints_%s.npy'%str(plant), views)

## 1.2 Reaname the file using the box coordinates

In [ ]:
save_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/new_name/plant%s/xyr_0_0_0'%str(plant)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

count = 0 
for h_ in range(box.shape[0]): 
    for w_ in range(box.shape[1]):
        if box[h_, w_, -1] is not None:
            count += 1
            pcd_new_name = f'pcd_{h_}_{w_}.pcd'
            rgb_new_name = f'rgb_{h_}_{w_}.jpeg'
            pose_new_name = f'pose_{h_}_{w_}.txt'

            base_name = box[h_, w_, -1].split('.')[0].split('_')[0]
            pcd_old_path = os.path.join(plant_data_dir, base_name + '_b.pcd')
            rgb_old_path = os.path.join(plant_data_dir, base_name + '_color.jpeg')
            pose_old_path = os.path.join(plant_data_dir, base_name + '_pose.txt')

            # copy the file to the new directory with the new name
            pcd_new_path = os.path.join(save_dir, pcd_new_name)
            rgb_new_path = os.path.join(save_dir, rgb_new_name)
            pose_new_path = os.path.join(save_dir, pose_new_name)

            os.system(f'cp {pcd_old_path} {pcd_new_path}')
            os.system(f'cp {rgb_old_path} {rgb_new_path}')
            os.system(f'cp {pose_old_path} {pose_new_path}')
print(count)


## 1.3. Crop, Transform, denoise and downsample the partial point cloud

In [ ]:
import numpy as np
import os
import open3d as o3d

if __name__ == '__main__':
    voxel_size = 0.003
    plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
    gt_path = DATA_ROOT + '/real_world_data_new/real_gt_pcd'
    dir_path = DATA_ROOT + '/real_world_data_new/real_nbv_data'

    ########################process partial point cloud########################
    for plant in plant_list:
        gt_pcd_path = os.path.join(gt_path, 'plant%s/tomato%s.ply'%(str(plant), str(plant)))
        gt_pcd = o3d.io.read_point_cloud(gt_pcd_path)

        viewpoints = np.load(os.path.join(dir_path, 'viewpoints_%s.npy'%str(plant)), allow_pickle=True)

        plant_folder = os.path.join(dir_path, 'new_name', 'plant%s'%str(plant), 'xyzr_0_0_0_0')
        files = os.listdir(plant_folder)
        pcds = [file for file in files if file.endswith('.pcd')]

        for pcd_name in pcds:
            pcd_path = os.path.join(plant_folder, pcd_name)
            pcd = o3d.io.read_point_cloud(pcd_path)

            base_name = pcd_name.split('.')[0]
            h, w = base_name.split('_')[1], base_name.split('_')[2]

            pose = viewpoints[int(h), int(w), :]
            pose = pose_to_transformation(pose)

            pcd.transform(pose)
            pcd_down_sampled = downsample_point_cloud(pcd, bbox=bbox, voxel_size=voxel_size)

            #visualize the point cloud
            # vis_PC(center, complete_pcd=None, partial_pcd=pcd_down_sampled, viewpoints_pcd=None, bbox=bbox)
            
            points = np.asarray(pcd_down_sampled.points)[np.newaxis, :, :]
            colors = np.asarray(pcd_down_sampled.colors)[np.newaxis, :, :]
            pcd_xyzrgb = np.concatenate([points, colors], axis=-1)
            # save the points in the same folder with the same base name but with .npy extension
            np.save(os.path.join(plant_folder, base_name+'.npy'), pcd_xyzrgb)

            print(pcd_xyzrgb.shape)

## 1.4.1 Get organized point clouds and cropped images

In [ ]:
import numpy as np
import os
import open3d as o3d
import cv2

if __name__ == '__main__':
    # Intel RealSense L515 camera intrinsics
    # CORRECTED: Original values were for different resolution, divided by 2 for 960x540
    camera_intrinsics = {
        'fx': 1135.306761618773 / 2.0,  # focal length x (was 2x too large)
        'fy': 1135.306761618773 / 2.0,  # focal length y (was 2x too large)
        'cx': 480.5,                     # principal point x
        'cy': 270.5                      # principal point y
    }
    image_width = 960
    image_height = 540
    
    plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
    dir_path = DATA_ROOT + '/real_world_data_new/real_nbv_data'

    # Depth crop threshold (meters): remove pixels with z > depth_max_m
    depth_max_m = 1.3

    def build_organized_xyzrgb(xyz_src, rgb_src, fx, fy, cx, cy, image_width, image_height):
        valid_depth = xyz_src[:, 2] > 0
        u = np.round(fx * xyz_src[:, 0] / xyz_src[:, 2] + cx).astype(np.int32)
        v = np.round(fy * xyz_src[:, 1] / xyz_src[:, 2] + cy).astype(np.int32)
        valid_u = (u >= 0) & (u < image_width)
        valid_v = (v >= 0) & (v < image_height)
        valid_mask = valid_depth & valid_u & valid_v
        depth_map = np.full((image_height, image_width), np.inf, dtype=np.float32)
        organized_xyzrgb = np.full((image_height, image_width, 6), np.nan, dtype=np.float32)
        if np.sum(valid_mask) == 0:
            return organized_xyzrgb, depth_map, 0
        idx_all = np.where(valid_mask)[0]
        u_valid = u[idx_all]
        v_valid = v[idx_all]
        z_valid = xyz_src[idx_all, 2]
        for i in range(len(idx_all)):
            uu = u_valid[i]
            vv = v_valid[i]
            z = z_valid[i]
            if z < depth_map[vv, uu]:
                depth_map[vv, uu] = z
                organized_xyzrgb[vv, uu, :3] = xyz_src[idx_all[i]]
                organized_xyzrgb[vv, uu, 3:] = rgb_src[idx_all[i]]
        return organized_xyzrgb, depth_map, len(idx_all)
    
    ########################process partial point cloud########################
    for plant in plant_list:
        plant_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'xyzr_0_0_0_0')
        
        # Check if input folder exists (created by Section 1.2)
        if not os.path.exists(plant_folder):
            print(f"SKIPPING plant{plant}: Input folder doesn't exist: {plant_folder}")
            print(f"  → Run Section 1.1 with plant={plant}, then run Section 1.2 first!")
            continue
        
        organized_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'organized_pcd_xyzr_0_0_0_0')
        pcd_rgb_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'pcd_cropped_rgb_xyzr_0_0_0_0')
        
        os.makedirs(organized_folder, exist_ok=True)
        os.makedirs(pcd_rgb_folder, exist_ok=True)
        
        files = os.listdir(plant_folder)
        pcds = [file for file in files if file.endswith('.pcd')]
        
        fx, fy = camera_intrinsics['fx'], camera_intrinsics['fy']
        cx, cy = camera_intrinsics['cx'], camera_intrinsics['cy']

        for pcd_name in pcds:
            pcd_path = os.path.join(plant_folder, pcd_name)
            pcd = o3d.io.read_point_cloud(pcd_path)
            
            base_name = pcd_name.split('.')[0]

            points = np.asarray(pcd.points)
            colors = np.asarray(pcd.colors)

            if points.shape[0] == 0:
                print(f"{pcd_name}: empty point cloud")
                continue

            # Project using raw camera-frame xyz (matches original RGB)
            organized_xyzrgb, depth_map, valid_count = build_organized_xyzrgb(
            points, colors, fx, fy, cx, cy, image_width, image_height
            )
            if valid_count == 0:
                print(f"{pcd_name}: no valid projections")
                continue

            # Save structured point cloud (H, W, 6)
            np.save(os.path.join(organized_folder, f'{base_name}_organized.npy'), organized_xyzrgb)

            # Create depth mask (crop by z > depth_max_m)
            valid_depth_mask = (depth_map <= depth_max_m) & np.isfinite(depth_map)

            # Extract and save RGB from point cloud projection (cropped by distance)
            pcd_rgb = (organized_xyzrgb[:, :, 3:6] * 255).astype(np.uint8)
            pcd_rgb[~valid_depth_mask] = 0
            pcd_rgb_bgr = cv2.cvtColor(pcd_rgb, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(pcd_rgb_folder, f'{base_name}_pcd_rgb.png'), pcd_rgb_bgr)

            # Debug: Check point cloud projection coverage
            valid_pixels = np.isfinite(depth_map)
            u_coords = np.round(fx * points[:, 0] / points[:, 2] + cx).astype(np.int32)
            v_coords = np.round(fy * points[:, 1] / points[:, 2] + cy).astype(np.int32)
            valid_proj = (points[:, 2] > 0) & (u_coords >= 0) & (u_coords < image_width) & (v_coords >= 0) & (v_coords < image_height)
            if np.sum(valid_proj) > 0:
                u_range = f"[{u_coords[valid_proj].min()}, {u_coords[valid_proj].max()}]"
                v_range = f"[{v_coords[valid_proj].min()}, {v_coords[valid_proj].max()}]"
                depth_range = f"[{points[valid_proj, 2].min():.3f}, {points[valid_proj, 2].max():.3f}]m"
                print(f"  PCD coverage - u: {u_range}, v: {v_range}, depth: {depth_range}")
                print(f"  Valid pixels: {np.sum(valid_pixels)} / {image_width*image_height} ({100*np.sum(valid_pixels)/(image_width*image_height):.1f}%)")

            print(f"Processed {pcd_name}: organized={organized_xyzrgb.shape}, pcd_rgb={pcd_rgb.shape}")

## 1.4.2 crop, transform and downsampling point cloud

In [ ]:
import numpy as np
import os
import cv2
import open3d as o3d

def voxel_downsample_with_class(points_xyzrgbc, voxel_size):
    if points_xyzrgbc.shape[0] == 0:
        return points_xyzrgbc
    xyz = points_xyzrgbc[:, :3]
    rgb = points_xyzrgbc[:, 3:6]
    cls = points_xyzrgbc[:, 6]
    voxel_idx = np.floor(xyz / voxel_size).astype(np.int32)
    voxel_dict = {}
    for i, v in enumerate(voxel_idx):
        key = (v[0], v[1], v[2])
        if key not in voxel_dict:
            voxel_dict[key] = []
        voxel_dict[key].append(i)
    down_points = []
    for idxs in voxel_dict.values():
        pts = xyz[idxs]
        cols = rgb[idxs]
        cls_vals = cls[idxs]
        xyz_mean = pts.mean(axis=0)
        rgb_mean = cols.mean(axis=0)
        cls_major = 1.0 if np.sum(cls_vals >= 0.5) >= (len(cls_vals) / 2.0) else 0.0
        down_points.append(np.concatenate([xyz_mean, rgb_mean, [cls_major]], axis=0))
    return np.asarray(down_points, dtype=np.float32)

def remove_statistical_outlier_xyzrgbc(points_xyzrgbc, nb_neighbors=20, std_ratio=0.2):
    if points_xyzrgbc.shape[0] == 0:
        return points_xyzrgbc
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_xyzrgbc[:, :3])
    _, ind = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    if len(ind) == 0:
        return points_xyzrgbc[:0]
    return points_xyzrgbc[np.asarray(ind)]

if __name__ == '__main__':
    plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]  # set as needed
    dir_path = DATA_ROOT + '/real_world_data_new/real_nbv_data'
    out_folder_name = 'xyzr_with_class_0_0_0_0'
    nb_neighbors = 20
    std_ratio = 0.2
    expand_radius = 0.006  # meters, expand class=1 to neighbors within this radius
    
    for plant in plant_list:
        viewpoints_path = os.path.join(dir_path, f'viewpoints_{plant}.npy')
        viewpoints = np.load(viewpoints_path, allow_pickle=True)
        organized_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'organized_pcd_xyzr_0_0_0_0')
        mask_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'masks')
        out_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', out_folder_name)
        os.makedirs(out_folder, exist_ok=True)

        organized_files = [f for f in os.listdir(organized_folder) if f.endswith('_organized.npy')]
        for organized_file in organized_files:
            organized_path = os.path.join(organized_folder, organized_file)
            organized_xyzrgb = np.load(organized_path)  # (H, W, 6)
            base_name = organized_file.replace('_organized.npy', '')

            # Load mask image (same HxW)
            mask_path = os.path.join(mask_folder, f'{base_name}_pcd_rgb.png')
            mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask_img is None:
                print(f"Missing mask: {mask_path}")
                continue
            if mask_img.shape[:2] != organized_xyzrgb.shape[:2]:
                mask_img = cv2.resize(mask_img, (organized_xyzrgb.shape[1], organized_xyzrgb.shape[0]), interpolation=cv2.INTER_NEAREST)
            class_map = (mask_img > 0).astype(np.float32)
            organized_xyzrgbc = np.concatenate([organized_xyzrgb, class_map[..., None]], axis=-1)

            # Keep only finite points
            finite_mask = np.isfinite(organized_xyzrgbc[..., 0]) & np.isfinite(organized_xyzrgbc[..., 1]) & np.isfinite(organized_xyzrgbc[..., 2])
            valid_points_xyzrgbc = organized_xyzrgbc[finite_mask]
            if valid_points_xyzrgbc.shape[0] == 0:
                print(f"{base_name}: no valid points")
                continue

            # Transform to world frame using viewpoint pose
            parts = base_name.split('_')
            if len(parts) >= 3:
                h, w = int(parts[1]), int(parts[2])
                pose = viewpoints[h, w, :]
                T = pose_to_transformation(pose)
                pts = valid_points_xyzrgbc[:, :3]
                pts_h = np.concatenate([pts, np.ones((pts.shape[0], 1), dtype=np.float32)], axis=1)
                pts_w = (T @ pts_h.T).T[:, :3]
                valid_points_xyzrgbc[:, :3] = pts_w
            else:
                print(f"{base_name}: cannot parse h,w for pose")
                continue

            # Crop by bbox in world frame
            min_b = bbox.get_min_bound()
            max_b = bbox.get_max_bound()
            in_box = (valid_points_xyzrgbc[:, 0] >= min_b[0]) & (valid_points_xyzrgbc[:, 0] <= max_b[0]) & \
                     (valid_points_xyzrgbc[:, 1] >= min_b[1]) & (valid_points_xyzrgbc[:, 1] <= max_b[1]) & \
                     (valid_points_xyzrgbc[:, 2] >= min_b[2]) & (valid_points_xyzrgbc[:, 2] <= max_b[2])
            valid_points_xyzrgbc = valid_points_xyzrgbc[in_box]
            if valid_points_xyzrgbc.shape[0] == 0:
                print(f"{base_name}: no points after bbox crop")
                continue

            # Expand class=1 to nearby points within expand_radius
            class_vals = valid_points_xyzrgbc[:, 6].copy()
            target_idx = np.where(class_vals >= 0.5)[0]
            if target_idx.size > 0 and expand_radius > 0:
                pcd_expand = o3d.geometry.PointCloud()
                pcd_expand.points = o3d.utility.Vector3dVector(valid_points_xyzrgbc[:, :3])
                kd_expand = o3d.geometry.KDTreeFlann(pcd_expand)
                for idx in target_idx:
                    _, idxs, _ = kd_expand.search_radius_vector_3d(valid_points_xyzrgbc[idx, :3], expand_radius)
                    if len(idxs) > 0:
                        class_vals[np.asarray(idxs)] = 1.0
                valid_points_xyzrgbc[:, 6] = class_vals

            # Voxel downsample while preserving class label
            points_down = voxel_downsample_with_class(valid_points_xyzrgbc, voxel_size)
            if points_down.shape[0] == 0:
                print(f"{base_name}: no points after voxel downsample")
                continue

            # Statistical outlier removal (preserve xyzrgbc)
            points_down = remove_statistical_outlier_xyzrgbc(
                points_down, nb_neighbors=nb_neighbors, std_ratio=std_ratio
            )
            if points_down.shape[0] == 0:
                print(f"{base_name}: no points after outlier removal")
                continue

            # Save as (1, N, 7) xyzrgbc
            out_path = os.path.join(out_folder, f'{base_name}.npy')
            np.save(out_path, points_down[np.newaxis, ...])
            print(f"Saved {out_path}: {points_down.shape}")

## 1.4.3 visualize the point cloud with semantic segmentation

In [ ]:
# visualize the point cloud with semantic segmentation - ALL views merged
import numpy as np
import open3d as o3d
import os

# Load all point clouds for a plant and merge them
plant = 37
dir_path = DATA_ROOT + '/real_world_data_new/real_nbv_data'
organized_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'xyzr_with_class_0_0_0_0')

# Debug: load and merge original point clouds from xyzr_0_0_0_0
debug_folder = os.path.join(dir_path, 'new_name', f'plant{plant}', 'xyzr_0_0_0_0')
pcd_debug = None
if os.path.exists(debug_folder):
    debug_files = [f for f in os.listdir(debug_folder) if f.startswith('pcd_') and f.endswith('.npy')]
    debug_xyz_all = []
    for df in debug_files:
        debug_path = os.path.join(debug_folder, df)
        debug_data = np.load(debug_path)  # (1, n, 6)
        debug_xyz_all.append(debug_data[0, :, :3])
    if len(debug_xyz_all) > 0:
        debug_xyz = np.concatenate(debug_xyz_all, axis=0)
        pcd_debug = o3d.geometry.PointCloud()
        pcd_debug.points = o3d.utility.Vector3dVector(debug_xyz)
        pcd_debug.paint_uniform_color([0, 0, 1])  # Blue for debug
    else:
        print(f"No debug pcd_*.npy found in: {debug_folder}")
else:
    print(f"Debug folder not found: {debug_folder}")

# Get all .npy files in the folder
if os.path.exists(organized_folder):
    pcd_files = [f for f in os.listdir(organized_folder) if f.endswith('.npy')]
    print(f"Found {len(pcd_files)} point cloud files for plant {plant}")
    
    # Load and merge all point clouds
    all_xyz = []
    all_rgb = []
    all_mask = []
    
    for pcd_file in pcd_files:
        pcd_path = os.path.join(organized_folder, pcd_file)
        pcd_data = np.load(pcd_path)  # (1, n, 7)
        
        all_xyz.append(pcd_data[0, :, :3])
        all_rgb.append(pcd_data[0, :, 3:6])
        all_mask.append(pcd_data[0, :, 6])
    
    # Concatenate all views
    xyz = np.concatenate(all_xyz, axis=0)
    rgb = np.concatenate(all_rgb, axis=0)
    mask = np.concatenate(all_mask, axis=0)
    
    print(f"\nMerged point cloud:")
    print(f"  Total points: {xyz.shape[0]}")
    print(f"  Mask range: [{mask.min():.3f}, {mask.max():.3f}]")
    print(f"  Mask distribution:")
    print(f"    mask == 0: {np.sum(mask == 0)} ({100*np.sum(mask == 0)/len(mask):.1f}%)")
    print(f"    mask == 1: {np.sum(mask == 1)} ({100*np.sum(mask == 1)/len(mask):.1f}%)")
    if pcd_debug is not None:
        debug_points = np.asarray(pcd_debug.points)
        debug_points = debug_points[np.isfinite(debug_points).all(axis=1)]
        debug_count = debug_points.shape[0]
        print(f"\nDebug point cloud:")
        print(f"  Total points: {debug_count}")
        print(f"  Debug/Merged ratio: {debug_count / max(1, xyz.shape[0]):.4f}")

        # Coverage: % debug points within threshold of merged cloud
        coverage_thr = 0.003
        xyz_kd = xyz[np.isfinite(xyz).all(axis=1)]
        if xyz_kd.shape[0] == 0 or debug_count == 0:
            print("  Coverage: skipped (empty or non-finite points)")
        else:
            pcd_kd = o3d.geometry.PointCloud()
            pcd_kd.points = o3d.utility.Vector3dVector(xyz_kd)
            kdtree = o3d.geometry.KDTreeFlann(pcd_kd)
            covered = 0
            for p in debug_points:
                _, _, dist2 = kdtree.search_knn_vector_3d(p, 1)
                if len(dist2) > 0 and dist2[0] <= coverage_thr * coverage_thr:
                    covered += 1
            coverage_rate = covered / max(1, debug_count)
            print(f"  Coverage (@{coverage_thr}m): {covered}/{debug_count} ({coverage_rate*100:.2f}%)")
    
    # Create three versions for visualization
    # 1. Original with RGB colors
    pcd_original = o3d.geometry.PointCloud()
    pcd_original.points = o3d.utility.Vector3dVector(xyz)
    pcd_original.colors = o3d.utility.Vector3dVector(rgb)
    
    # 2. Binary segmentation (red=tomato, green=background)
    pcd_seg = o3d.geometry.PointCloud()
    pcd_seg.points = o3d.utility.Vector3dVector(xyz)
    colors_seg = np.zeros_like(rgb)
    segmented_mask = mask > 0
    colors_seg[segmented_mask] = [1, 0, 0]  # Red for tomatoes
    colors_seg[~segmented_mask] = [0, 1, 0]  # Green for background
    pcd_seg.colors = o3d.utility.Vector3dVector(colors_seg)
    
    # 3. Blend: original color with red overlay
    pcd_blend = o3d.geometry.PointCloud()
    pcd_blend.points = o3d.utility.Vector3dVector(xyz)
    colors_blend = rgb.copy()
    colors_blend[:, 0] = np.clip(colors_blend[:, 0] + mask * 0.7, 0, 1)
    pcd_blend.colors = o3d.utility.Vector3dVector(colors_blend)
    
    # Visualize
    print("\nVisualization options:")
    print("1. Organized + reference (xyzrgbc + xyzr_0_0_0_0)")
    print("2. Binary segmentation (red=tomato, green=background)")
    print("3. Blended (original + red overlay)")
    print("4. Reference only (xyzr_0_0_0_0)")
    
    coord_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
    
    # Option 1: Organized (xyzrgbc) + reference (xyzr_0_0_0_0)
    # if pcd_debug is not None:
    #     geoms1 = [pcd_original, pcd_debug, coord_frame]
    #     o3d.visualization.draw_geometries(geoms1,
    #                                       window_name=f'Plant {plant} - Organized + Reference',
    #                                       width=800, height=600)
    # else:
    #     geoms1 = [pcd_original, coord_frame]
    #     o3d.visualization.draw_geometries(geoms1,
    #                                       window_name=f'Plant {plant} - Organized Only',
    #                                       width=800, height=600)
    
    # # Option 4: Reference only (xyzr_0_0_0_0)
    # if pcd_debug is not None:
    #     geoms_ref = [pcd_debug, coord_frame]
    #     o3d.visualization.draw_geometries(geoms_ref,
    #                                       window_name=f'Plant {plant} - Reference Only (xyzr_0_0_0_0)',
    #                                       width=800, height=600)
    # else:
    #     print("Reference point cloud not available; nothing to show.")
    
    # Option 2: Binary segmentation
    geoms2 = [pcd_seg, coord_frame]
    if pcd_debug is not None:
        geoms2.append(pcd_debug)
    o3d.visualization.draw_geometries(geoms2, 
                                      window_name=f'Plant {plant} - Segmentation (All Views)',
                                      width=800, height=600)
    
    # Option 3: Blended
    # geoms3 = [pcd_blend, coord_frame]
    # if pcd_debug is not None:
    #     geoms3.append(pcd_debug)
    # o3d.visualization.draw_geometries(geoms3, 
    #                                   window_name=f'Plant {plant} - Blended (All Views)',
    #                                   width=800, height=600)
else:
    print(f"Folder not found: {organized_folder}")
    print("Run section 1.4 first to generate point clouds with masks")

# 2. Process gt pcd

# 2.1. Crop and denoise the .ply for gt point cloud, then store as .npy

In [ ]:
import numpy as np
import os
import open3d as o3d
from scipy.spatial.transform import Rotation as R

if __name__ == '__main__':
    voxel_size = 0.003
    plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
    dir_path = DATA_ROOT + '/real_world_data_new/real_gt_pcd'
    
    ########################process complete point cloud########################
    for plant in plant_list:
        pcd_name = 'tomato%s.ply'%str(plant)
        pcd_path = os.path.join(dir_path, 'plant%s'%str(plant), pcd_name)
        pcd = o3d.io.read_point_cloud(pcd_path)
        base_name = pcd_name.split('.')[0]
        pcd_down_sampled = downsample_point_cloud(pcd, bbox, voxel_size)
        #visualize the point cloud
        # vis_PC(center, complete_pcd=pcd_down_sampled, partial_pcd=None, viewpoints_pcd=None, bbox=bbox)
        pcd_down_sampled_points = np.asarray(pcd_down_sampled.points)[np.newaxis, :, :]
        pcd_down_sampled_colors = np.asarray(pcd_down_sampled.colors)[np.newaxis, :, :]
        
        pcd_down_sampled_points_colors = np.concatenate([pcd_down_sampled_points, pcd_down_sampled_colors], axis=-1)
        # save the points in the same folder with the same base name but with .npy extension
        np.save(os.path.join(dir_path,'plant%s'%str(plant) ,'downsampled_pcd_0_0_0.npy'), pcd_down_sampled_points_colors)



# 2.2. Align gt to partial point clouds

In [ ]:
# Fixed point cloud alignment: GT to accumulated point cloud
import numpy as np
import open3d as o3d
import copy

def prepare_point_cloud_for_icp(pcd, voxel_size=0.003):
    """Prepare point cloud for ICP by computing normals and cleaning up"""
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    return pcd
# read the gt pcd
plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
for plant in plant_list:
    print(f"\n=== Processing Plant {plant} ===")
    gt_dir = DATA_ROOT + '/real_world_data_new/real_gt_pcd'
    gt_pcd = np.load(os.path.join(gt_dir, 'plant%s'%str(plant), 'downsampled_pcd_0_0_0.npy'))
    gt_pcd_o3d = o3d.geometry.PointCloud()
    gt_pcd_o3d.points = o3d.utility.Vector3dVector(gt_pcd[0, :, :3])
    gt_pcd_o3d.colors = o3d.utility.Vector3dVector(gt_pcd[0, :, 3:])
    # gt_pcd_o3d.paint_uniform_color([0.5, 0.5, 0.5])  # grey for GT

    partial_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/new_name/plant%s/xyzr_0_0_0_0'%str(plant)
    pcd_list = os.listdir(partial_dir)
    pcd_list = [f for f in pcd_list if f.endswith('.npy')]

    acc_ocd = []
    for pcd_name in pcd_list:
        pcd = np.load(os.path.join(partial_dir, pcd_name))
        acc_ocd.append(pcd)

    acc_ocd = np.concatenate(acc_ocd, axis=1)
    acc_ocd_o3d = o3d.geometry.PointCloud()
    acc_ocd_o3d.points = o3d.utility.Vector3dVector(acc_ocd[0,:,:3])
    acc_ocd_o3d.colors = o3d.utility.Vector3dVector(acc_ocd[0,:,3:])

    acc_ocd_o3d = downsample_point_cloud(acc_ocd_o3d, bbox=bbox, voxel_size=voxel_size)

    # Prepare both point clouds for ICP (compute normals, remove outliers)
    print("Preparing point clouds for alignment...")
    acc_ocd_prepared = prepare_point_cloud_for_icp(copy.deepcopy(acc_ocd_o3d), voxel_size=voxel_size)
    gt_pcd_prepared = prepare_point_cloud_for_icp(copy.deepcopy(gt_pcd_o3d), voxel_size=voxel_size)
    
    # Method 1: Point-to-Point ICP (more robust, doesn't require normals)
    print("Performing Point-to-Point ICP alignment...")
    icp_p2p = o3d.pipelines.registration.registration_icp(
        gt_pcd_prepared, acc_ocd_prepared, voxel_size, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint()
    )
    
    # Method 2: Point-to-Plane ICP (now with normals computed)
    print("Performing Point-to-Plane ICP alignment...")
    icp_p2plane = o3d.pipelines.registration.registration_icp(
        gt_pcd_prepared, acc_ocd_prepared, voxel_size, icp_p2p.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )
    
    print(f"Point-to-Point ICP - Fitness: {icp_p2p.fitness:.4f}, RMSE: {icp_p2p.inlier_rmse:.4f}")
    print(f"Point-to-Plane ICP - Fitness: {icp_p2plane.fitness:.4f}, RMSE: {icp_p2plane.inlier_rmse:.4f}")
    
    # Apply transformation to GT point cloud (align GT to accumulated)
    gt_pcd_aligned = copy.deepcopy(gt_pcd_o3d)
    gt_pcd_aligned.transform(icp_p2plane.transformation)
    # gt_pcd_aligned.paint_uniform_color([0, 0, 1])  # Blue for aligned GT

    # save the aligned gt_pcd on the same folder as gt_pcd but with the name of plant_aligned.npy
    gt_pcd_aligned_points = np.asarray(gt_pcd_aligned.points)[np.newaxis, :, :]
    gt_pcd_aligned_colors = np.asarray(gt_pcd_aligned.colors)[np.newaxis, :, :]
    gt_pcd_aligned_points_colors = np.concatenate([gt_pcd_aligned_points, gt_pcd_aligned_colors], axis=-1)
    # np.save(os.path.join(gt_dir, f'plant{plant}_aligned.npy'), gt_pcd_aligned_points_colors)
    # store the point cloud as .pcd
    o3d.io.write_point_cloud(os.path.join(gt_dir, 'plant%s.pcd'%str(plant)), gt_pcd_aligned)


    

    


# 3. Final test of coverage

In [ ]:
# usr/bin/env python3
import numpy as np
import os
import open3d as o3d
from scipy.spatial.transform import Rotation as R
import torch
# add this path to the python path
import sys
sys.path.append(pkg_path('networks') + '/src/networks')
from chamfer import chamfer_distance

def cal_coverage(partial_pc, complete_pc, thr=0.003):
    '''
        partial_pc: 6 channels-xyzrgb.
    '''
    if not partial_pc.is_cuda:
        partial_pc = partial_pc.to('cuda')
    _, dist2 = get_chamfer_dists(partial_pc[..., 0:3], complete_pc)
    covered_points = dist2 < thr
    cov_ratio = torch.sum(covered_points).item() / covered_points.shape[1]
    return '%s/%s' % (str(torch.sum(covered_points).item()),
                        str(covered_points.shape[1])), cov_ratio

def get_chamfer_dists(pcd_x, pcd_y):
    # We compare the two sets of pointclouds by computing the chamfer loss.
    _, _, dist_x, dist_y = chamfer_distance(pcd_x, pcd_y)
    dist_x = dist_x.cpu().detach()
    dist_y = dist_y.cpu().detach()

    return dist_x, dist_y
# read the gt pcd
plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]

for plant in plant_list:
    gt_pcd = np.load(DATA_ROOT + '/real_world_data_new/real_gt_pcd/plant%s/plant%s_aligned.npy'%(str(plant), str(plant)))
    print(gt_pcd.shape)
    gt_pcd_o3d = o3d.geometry.PointCloud()
    gt_pcd_o3d.points = o3d.utility.Vector3dVector(gt_pcd[0, :, :3])
    gt_pcd_o3d.paint_uniform_color([0.5, 0.5, 0.5])  # grey

    partial_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/new_name/plant%s/xyr_0_0_0'%str(plant)
    pcd_list = os.listdir(partial_dir)
    pcd_list = [f for f in pcd_list if f.endswith('.npy')]

    acc_pcd = []
    for pcd_name in pcd_list:
        pcd = np.load(os.path.join(partial_dir, pcd_name))
        acc_pcd.append(pcd)

    acc_pcd = np.concatenate(acc_pcd, axis=1)
    acc_pcd_o3d = o3d.geometry.PointCloud()
    acc_pcd_o3d.points = o3d.utility.Vector3dVector(acc_pcd[0,:,:3])  # Fixed: use acc_pcd instead of acc_ocd
    acc_pcd_o3d.paint_uniform_color([0, 1, 0])  # green

    # Downsample accumulated point cloud
    acc_pcd_o3d = downsample_point_cloud(acc_pcd_o3d, bbox=bbox, voxel_size=voxel_size)
    
    # Convert to numpy arrays to get shape information
    acc_pcd_points = np.asarray(acc_pcd_o3d.points)
    
    print(f"Number of points - Accumulated: {len(acc_pcd_o3d.points)}, GT: {gt_pcd.shape}")

    # # calculate the coverage
    print(torch.tensor(acc_pcd_points).cuda().unsqueeze(0).shape)
    print(torch.tensor(gt_pcd).cuda().shape)
    
    cal = cal_coverage(torch.from_numpy(acc_pcd_points).cuda().unsqueeze(0), torch.from_numpy(gt_pcd[..., :3]).cuda(), thr=voxel_size)
    print(cal)
    
    # Optional: Visualize the point clouds
    o3d.visualization.draw_geometries([acc_pcd_o3d, gt_pcd_o3d], 
                                      window_name=f'Plant {plant} - Before Alignment', 
                                      width=800, height=600)


# 4. Align annotation as well

In [ ]:
# Fixed point cloud alignment: GT to accumulated point cloud
import numpy as np
import open3d as o3d
import copy
import pandas as pd
import os

def prepare_point_cloud_for_icp(pcd, voxel_size=0.003):
    """Prepare point cloud for ICP by computing normals and cleaning up"""
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    return pcd

# read the gt pcd
plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
for plant in plant_list:
    print(f"\n=== Processing Plant {plant} ===")
    gt_dir = DATA_ROOT + '/real_world_data_new/real_gt_pcd'
    gt_pcd = np.load(os.path.join(gt_dir, 'plant%s'%str(plant), 'downsampled_pcd_0_0_0.npy'))
    gt_pcd_o3d = o3d.geometry.PointCloud()
    gt_pcd_o3d.points = o3d.utility.Vector3dVector(gt_pcd[0, :, :3])
    gt_pcd_o3d.colors = o3d.utility.Vector3dVector(gt_pcd[0, :, 3:])
    # gt_pcd_o3d.paint_uniform_color([0.5, 0.5, 0.5])  # grey for GT

    partial_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/new_name/plant%s/xyr_0_0_0'%str(plant)
    pcd_list = os.listdir(partial_dir)
    pcd_list = [f for f in pcd_list if f.endswith('.npy')]

    annotaiton_dir = DATA_ROOT + '/real_world_data_new/real_anno'
    # read the annotation file with only read
    annotation_file = pd.read_excel(os.path.join(annotaiton_dir, 'plant%s_object_poses.xlsx'%str(plant)) )

    acc_ocd = []
    for pcd_name in pcd_list:
        pcd = np.load(os.path.join(partial_dir, pcd_name))
        acc_ocd.append(pcd)

    acc_ocd = np.concatenate(acc_ocd, axis=1)
    acc_ocd_o3d = o3d.geometry.PointCloud()
    acc_ocd_o3d.points = o3d.utility.Vector3dVector(acc_ocd[0,:,:3])
    acc_ocd_o3d.paint_uniform_color([0, 1, 0])  # Green for accumulated

    acc_ocd_o3d = downsample_point_cloud(acc_ocd_o3d, bbox=bbox, voxel_size=voxel_size)

    # Prepare both point clouds for ICP (compute normals, remove outliers)
    print("Preparing point clouds for alignment...")
    acc_ocd_prepared = prepare_point_cloud_for_icp(copy.deepcopy(acc_ocd_o3d), voxel_size=voxel_size)
    gt_pcd_prepared = prepare_point_cloud_for_icp(copy.deepcopy(gt_pcd_o3d), voxel_size=voxel_size)
    
    # Method 1: Point-to-Point ICP (more robust, doesn't require normals)
    print("Performing Point-to-Point ICP alignment...")
    icp_p2p = o3d.pipelines.registration.registration_icp(
        gt_pcd_prepared, acc_ocd_prepared, voxel_size, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint()
    )
    
    # Method 2: Point-to-Plane ICP (now with normals computed)
    print("Performing Point-to-Plane ICP alignment...")
    icp_p2plane = o3d.pipelines.registration.registration_icp(
        gt_pcd_prepared, acc_ocd_prepared, voxel_size, icp_p2p.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )
    
    print(f"Point-to-Point ICP - Fitness: {icp_p2p.fitness:.4f}, RMSE: {icp_p2p.inlier_rmse:.4f}")
    print(f"Point-to-Plane ICP - Fitness: {icp_p2plane.fitness:.4f}, RMSE: {icp_p2plane.inlier_rmse:.4f}")
    
    # Apply transformation to GT point cloud (align GT to accumulated)
    gt_pcd_aligned = copy.deepcopy(gt_pcd_o3d)
    gt_pcd_aligned.transform(icp_p2plane.transformation)
    # gt_pcd_aligned.paint_uniform_color([0, 0, 1])  # Blue for aligned GT

    fruit_annotation_file_aligned = pd.DataFrame()
    node_annotation_file_aligned = pd.DataFrame()
    for index, row in annotation_file.iterrows():
        # get the name of the column
        bbox_name = row.object
        print(bbox_name)
        bbox_center = np.array([row['x'], row['y'], row['z']])
        bbox_center_aligned = icp_p2plane.transformation[:3, :3] @ bbox_center + icp_p2plane.transformation[:3, 3]
        if bbox_name.startswith('P'):
            node_annotation_file_aligned.at[index, 'loc_x'] = bbox_center_aligned[0]
            node_annotation_file_aligned.at[index, 'loc_y'] = bbox_center_aligned[1]
            node_annotation_file_aligned.at[index, 'loc_z'] = bbox_center_aligned[2]
            node_annotation_file_aligned.at[index, 'dir_x'] = 0
            node_annotation_file_aligned.at[index, 'dir_y'] = 0
            node_annotation_file_aligned.at[index, 'dir_z'] = 1
        elif bbox_name.startswith('F'):
            fruit_annotation_file_aligned.at[index, 'loc_x'] = bbox_center_aligned[0]
            fruit_annotation_file_aligned.at[index, 'loc_y'] = bbox_center_aligned[1]
            fruit_annotation_file_aligned.at[index, 'loc_z'] = bbox_center_aligned[2]
            fruit_annotation_file_aligned.at[index, 'radius'] = 0.06/2

    # save the annotation_file_aligned
    fruit_annotation_file_aligned.to_csv(os.path.join(annotaiton_dir, 'plant%s_export_fruit.csv'%str(plant)), index=False)
    node_annotation_file_aligned.to_csv(os.path.join(annotaiton_dir, 'plant%s_export_node.csv'%str(plant)), index=False)


# 5. Visualize annoation BBox

In [ ]:
import pandas as pd
import numpy as np
import open3d as o3d
import os

plant_list = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40] # 32 has no fruit bbox
# give me 11 unique colors in rgb order # green, blue, red, yellow, purple, cyan, brown, pink, orange, gray, black
colors = [[0, 1, 0], [0, 0, 1], [1, 0, 0], [1, 1, 0], [1, 0, 1], [0, 1, 1], [0.5, 0.5, 0], [0.5, 0, 0.5], [1, 0.5, 0], [0.5, 0.5, 0.5], [0, 0, 0]]

for plant in plant_list:
    gt_pcd = np.load(DATA_ROOT + '/real_world_data_new/real_gt_pcd/plant%s/plant%s_aligned.npy'%(str(plant), str(plant)))
    print(gt_pcd.shape)
    gt_pcd_o3d = o3d.geometry.PointCloud()
    gt_pcd_o3d.points = o3d.utility.Vector3dVector(gt_pcd[0, :, :3])
    # gt_pcd_o3d.colors = o3d.utility.Vector3dVector(gt_pcd[0, :, 3:])
    gt_pcd_o3d.paint_uniform_color([0.5, 0.5, 0.5])

    partial_dir = DATA_ROOT + '/real_world_data_new/real_nbv_data/new_name/plant%s/xyzr_0_0_0_0'%str(plant)
    pcd_list = os.listdir(partial_dir)
    pcd_list = [f for f in pcd_list if f.endswith('.npy')]

    acc_pcd = []
    for pcd_name in pcd_list:
        pcd = np.load(os.path.join(partial_dir, pcd_name))
        acc_pcd.append(pcd)

    acc_pcd = np.concatenate(acc_pcd, axis=1)
    acc_pcd_o3d = o3d.geometry.PointCloud()
    acc_pcd_o3d.points = o3d.utility.Vector3dVector(acc_pcd[0, :,:3])  # Fixed: use acc_pcd instead of acc_ocd
    acc_pcd_o3d.colors = o3d.utility.Vector3dVector(acc_pcd[0, :, 3:])
    # acc_pcd_o3d.paint_uniform_color([0, 1, 0])

    # Downsample accumulated point cloud
    acc_pcd_o3d = downsample_point_cloud(acc_pcd_o3d, bbox=bbox, voxel_size=voxel_size)

    dim = 0.06
    fruit_bbox_list = []
    node_bbox_list = []
    try:
        fruit_poses  =  pd.read_csv(DATA_ROOT + '/real_world_data_new/real_anno/plant%s_export_fruit.csv'%(str(plant)))
        fruit_poses = fruit_poses[['loc_x', 'loc_y', 'loc_z']].values

        for idx,  pose in enumerate(fruit_poses):   
            fruit_bbox_o3d = o3d.geometry.AxisAlignedBoundingBox(min_bound=pose-dim/2, max_bound=pose+dim/2)
            fruit_bbox_o3d.color = colors[idx]
            fruit_bbox_list.append(fruit_bbox_o3d)
    except:
        print(f'Plant {plant} has no fruit bbox')
    
    try:
        node_poses = pd.read_csv(DATA_ROOT + '/real_world_data_new/real_anno/plant%s_export_internode.csv'%(str(plant)))
        node_poses = node_poses[['loc_x', 'loc_y', 'loc_z']].values
      
        for idx, pose in enumerate(node_poses):
            node_bbox_o3d = o3d.geometry.AxisAlignedBoundingBox(min_bound=pose-dim/2, max_bound=pose+dim/2)
            node_bbox_o3d.color = colors[idx]
            node_bbox_list.append(node_bbox_o3d)
    except:
        print(f'Plant {plant} has no node bbox')
    
    coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
    # Optional: Visualize the point clouds
    o3d.visualization.draw_geometries([acc_pcd_o3d, *fruit_bbox_list, *node_bbox_list, coordinate_frame], 
                                      window_name=f'Plant {plant} - Aligned', 
                                      width=800, height=600)
